<a href="https://colab.research.google.com/github/amitpaudel21/BIO_INSPIRED_SYSTEM/blob/main/1BF24CS042_BIS_PARTICLE_SWARM_OPTIMIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random
import numpy as np

# 1. Problem parameters
HOURS = 24

# Electricity price for each hour
PRICE = np.array([
    0.10, 0.09, 0.08, 0.08, 0.09, 0.12,
    0.18, 0.25, 0.30, 0.28, 0.22, 0.20,
    0.18, 0.18, 0.20, 0.24, 0.32, 0.38,
    0.40, 0.36, 0.28, 0.22, 0.16, 0.12
])

# Each appliance has:
# name, power (kW), duration (hours),
# earliest start, latest start
APPLIANCES = [
    {"name": "Dishwasher", "power": 1.2,
     "duration": 2, "earliest": 0, "latest": 8},

    {"name": "Washing Machine", "power": 0.5,
     "duration": 2, "earliest": 6, "latest": 14},

    {"name": "Clothes Dryer", "power": 2.0,
     "duration": 1, "earliest": 8, "latest": 17},

    {"name": "Water Heater", "power": 1.5,
     "duration": 2, "earliest": 4, "latest": 18},

    {"name": "Pool Filter", "power": 0.8,
     "duration": 4, "earliest": 0, "latest": 16},

    {"name": "Dehumidifier", "power": 0.2,
     "duration": 5, "earliest": 0, "latest": 19}
]

# Smart battery parameters
BATTERY_CAPACITY = 13.5  # kWh
BATTERY_MAX_POWER = 5.0  # kW
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95

# PSO parameters
SWARM_SIZE = 20
MAX_ITERATIONS = 250
C1 = 2.0
C2 = 2.0
W_MAX = 1.0
W_MIN = 0.4
V_MAX = 4.0
V_MIN = -4.0


def valid_start(value, appliance):
    """Round and restrict the start time to its valid window."""
    return int(np.clip(
        round(value),
        appliance["earliest"],
        appliance["latest"]
    ))


def calculate_load(schedule):
    """Calculate total appliance power at every hour."""
    load = np.zeros(HOURS)

    for start, appliance in zip(schedule, APPLIANCES):
        end = start + appliance["duration"]
        load[start:end] += appliance["power"]

    return load


def operate_battery(load, use_battery):
    """Simulate battery charging and discharging."""
    grid_load = load.copy()

    if not use_battery:
        return grid_load

    stored_energy = 0.0
    average_price = np.mean(PRICE)

    for hour in range(HOURS):

        # Discharge during expensive hours.
        if PRICE[hour] > average_price and stored_energy > 0:
            discharge = min(
                BATTERY_MAX_POWER,
                load[hour],
                stored_energy * DISCHARGE_EFFICIENCY
            )

            grid_load[hour] -= discharge
            stored_energy -= (
                discharge / DISCHARGE_EFFICIENCY
            )

        # Charge during inexpensive hours.
        elif (
            PRICE[hour] < average_price
            and stored_energy < BATTERY_CAPACITY
        ):
            future_demand = sum(
                load[h]
                for h in range(hour + 1, HOURS)
                if PRICE[h] > average_price
            )

            target = min(BATTERY_CAPACITY, future_demand)
            needed = max(0.0, target - stored_energy)

            charge = min(
                BATTERY_MAX_POWER,
                needed / CHARGE_EFFICIENCY,
                (BATTERY_CAPACITY - stored_energy)
                / CHARGE_EFFICIENCY
            )

            grid_load[hour] += charge
            stored_energy += charge * CHARGE_EFFICIENCY

    return grid_load


def fitness(position, use_battery=True):
    """Objective: minimize the electricity bill only."""

    schedule = [
        valid_start(x, appliance)
        for x, appliance in zip(position, APPLIANCES)
    ]

    load = calculate_load(schedule)
    grid_load = operate_battery(load, use_battery)

    # Single objective: electricity cost
    bill = np.sum(grid_load * PRICE)

    return float(bill), schedule


def particle_swarm_optimization(use_battery=True):
    """Optimize appliance start times using PSO."""

    random.seed(7)
    np.random.seed(7)

    dimensions = len(APPLIANCES)

    # Initialize particle positions and velocities.
    positions = np.array([
        [
            random.uniform(a["earliest"], a["latest"])
            for a in APPLIANCES
        ]
        for _ in range(SWARM_SIZE)
    ])

    velocities = np.random.uniform(
        V_MIN, V_MAX, (SWARM_SIZE, dimensions)
    )

    personal_best = positions.copy()

    personal_fitness = np.array([
        fitness(p, use_battery)[0] for p in positions
    ])

    best_index = np.argmin(personal_fitness)
    global_best = personal_best[best_index].copy()
    global_fitness = float(personal_fitness[best_index])

    # Main PSO iterations
    for iteration in range(MAX_ITERATIONS):

        # Linearly decrease inertia weight.
        w = W_MAX - (
            (W_MAX - W_MIN)
            * iteration / MAX_ITERATIONS
        )

        for i in range(SWARM_SIZE):

            r1 = np.random.random(dimensions)
            r2 = np.random.random(dimensions)

            # Update velocity.
            velocities[i] = (
                w * velocities[i]
                + C1 * r1 * (personal_best[i] - positions[i])
                + C2 * r2 * (global_best - positions[i])
            )

            velocities[i] = np.clip(
                velocities[i], V_MIN, V_MAX
            )

            # Update position.
            positions[i] += velocities[i]

            # Enforce appliance operating windows.
            for j, appliance in enumerate(APPLIANCES):
                positions[i, j] = np.clip(
                    positions[i, j],
                    appliance["earliest"],
                    appliance["latest"]
                )

            current_fitness = fitness(
                positions[i], use_battery
            )[0]

            # Update personal best.
            if current_fitness < personal_fitness[i]:
                personal_fitness[i] = current_fitness
                personal_best[i] = positions[i].copy()

            # Update global best.
            if current_fitness < global_fitness:
                global_fitness = current_fitness
                global_best = positions[i].copy()

    return fitness(global_best, use_battery)


def main():
    # Optimize with and without the battery.
    without_battery = particle_swarm_optimization(False)
    with_battery = particle_swarm_optimization(True)

    print("POWER SCHEDULING USING PSO")
    print("=" * 48)

    print("\nOptimal schedule WITHOUT battery:")
    for start, appliance in zip(
        without_battery[1], APPLIANCES
    ):
        end = start + appliance["duration"]
        print(f"{appliance['name']:<20} {start:02d}:00 - {end:02d}:00")

    print(f"Electricity bill: {without_battery[0]:.4f}")

    print("\nOptimal schedule WITH battery:")
    for start, appliance in zip(
        with_battery[1], APPLIANCES
    ):
        end = start + appliance["duration"]
        print(f"{appliance['name']:<20} {start:02d}:00 - {end:02d}:00")

    print(f"Electricity bill: {with_battery[0]:.4f}")

    savings = without_battery[0] - with_battery[0]
    print(f"\nBill savings: {savings:.4f}")

    if without_battery[0] > 0:
        percentage = 100 * savings / without_battery[0]
        print(f"Percentage savings: {percentage:.2f}%")


if __name__ == "__main__":
    main()


POWER SCHEDULING USING PSO

Optimal schedule WITHOUT battery:
Dishwasher           02:00 - 04:00
Washing Machine      12:00 - 14:00
Clothes Dryer        13:00 - 14:00
Water Heater         04:00 - 06:00
Pool Filter          00:00 - 04:00
Dehumidifier         00:00 - 05:00
Electricity bill: 1.4150

Optimal schedule WITH battery:
Dishwasher           02:00 - 04:00
Washing Machine      09:00 - 11:00
Clothes Dryer        08:00 - 09:00
Water Heater         04:00 - 06:00
Pool Filter          00:00 - 04:00
Dehumidifier         00:00 - 05:00
Electricity bill: 1.2238

Bill savings: 0.1912
Percentage savings: 13.51%
